In [ ]:
%matplotlib widget

In [ ]:
import flammkuchen as fl
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import glob

import os

In [ ]:
from fimpylab import TwoPExperiment
from scipy.interpolate import interp1d

In [ ]:
def build_motion_regressors(stimulus_data_path, imaging_length=1831, imaging_rate=3, 
                     dataset_name=None, group_path='/'):
    """
    Build leftward and rightward motion regressors from stimulus data stored in HDF5 format.
    
    Parameters:
    -----------
    stimulus_data_path : str
        Path to the stimulus data file (.h5 or .hdf5)
    imaging_length : int
        Length of the imaging data (number of frames)
    imaging_rate : float
        Imaging frame rate in Hz
    dataset_name : str or None
        Name of the dataset within the h5 file. If None, attempt to load directly
    group_path : str
        Path to the group containing the dataset in the h5 file
    
    Returns:
    --------
    tuple
        (left_regressor, right_regressor, time_points)
    """
    import h5py  # Import here for more direct control
    
    # First, let's inspect the structure of the file to understand what we're working with
    try:
        with h5py.File(stimulus_data_path, 'r') as f:
            print(f"  HDF5 file structure:")
            
            # Helper function to print hierarchy
            def print_hdf5_structure(name, obj):
                if isinstance(obj, h5py.Dataset):
                    print(f"    Dataset: {name}, Shape: {obj.shape}, Type: {obj.dtype}")
                else:
                    print(f"    Group: {name}")
                return None
            
            # Print structure
            f.visititems(print_hdf5_structure)
            
            # Try to find structures that might contain our data
            coherence_data = None
            frozen_data = None
            time_data = None
            
            # Look for datasets with promising names
            for name, obj in f.items():
                if isinstance(obj, h5py.Dataset):
                    if 'coherence' in name.lower():
                        coherence_data = obj[:]
                        print(f"    Found coherence data in {name}")
                    elif 'frozen' in name.lower():
                        frozen_data = obj[:]
                        print(f"    Found frozen data in {name}")
                    elif 't' == name or 'time' in name.lower():
                        time_data = obj[:]
                        print(f"    Found time data in {name}")
            
            # If we found our required data directly
            if coherence_data is not None and frozen_data is not None:
                print("    Using directly found coherence and frozen datasets")
                stim_data = pd.DataFrame({
                    'random_dots_coherence': coherence_data,
                    'random_dots_frozen': frozen_data,
                    't': time_data if time_data is not None else np.arange(len(coherence_data))
                })
                
            # If we couldn't find direct matches, try to load complete datasets
            else:
                # Look for dataset with structure matching what we need
                for name, obj in f.items():
                    if isinstance(obj, h5py.Dataset) and obj.ndim > 0:
                        try:
                            print(f"    Attempting to load full dataset {name}")
                            # Try to convert to DataFrame with pandas
                            df = pd.DataFrame(obj[:])
                            
                            # Check if it has our columns of interest
                            required_cols = ['random_dots_coherence', 'random_dots_frozen']
                            if all(col in df.columns for col in required_cols):
                                print(f"    Found dataframe with required columns in {name}")
                                stim_data = df
                                break
                            else:
                                col_names = list(df.columns)
                                print(f"    Columns in {name}: {col_names}")
                        except Exception as e:
                            print(f"    Could not convert {name} to DataFrame: {str(e)}")
                
                # If still not found, try a different approach with flammkuchen
                if 'stim_data' not in locals():
                    try:
                        print("    Attempting to load with flammkuchen...")
                        all_data = fl.load(stimulus_data_path)
                        
                        # Try to find the data in the returned structure
                        if isinstance(all_data, dict):
                            for key, value in all_data.items():
                                if isinstance(value, dict) and 'random_dots_coherence' in value:
                                    stim_data = pd.DataFrame(value)
                                    print(f"    Found dataframe in dictionary key: {key}")
                                    break
                                elif isinstance(value, pd.DataFrame) and 'random_dots_coherence' in value.columns:
                                    stim_data = value
                                    print(f"    Found dataframe in key: {key}")
                                    break
                    except Exception as e:
                        print(f"    Flammkuchen loading failed: {str(e)}")
    
    except Exception as e:
        raise Exception(f"Failed to inspect HDF5 file: {str(e)}")
    
    # If we still don't have the data, raise an exception
    if 'stim_data' not in locals():
        raise Exception("Could not find required data in the HDF5 file")
    
    # Create basic regressors at the stimulus timescale
    left_regressor_raw = (stim_data['random_dots_coherence'] < 0) & (stim_data['random_dots_frozen'] == 0)
    right_regressor_raw = (stim_data['random_dots_coherence'] > 0) & (stim_data['random_dots_frozen'] == 0)
    
    # Convert boolean arrays to integers (0 and 1)
    left_regressor_raw = left_regressor_raw.astype(int)
    right_regressor_raw = right_regressor_raw.astype(int)
    
    # Get stimulus time points
    stim_time = stim_data['t'].values
    
    # Create new time points for imaging data (at 3 Hz)
    # Make sure the time span covers the entire stimulus presentation
    imaging_time = np.linspace(stim_time[0], stim_time[-1], imaging_length)
    
    # Interpolate regressors to match imaging frame rate
    left_interp = interp1d(stim_time, left_regressor_raw, kind='nearest', bounds_error=False, fill_value=0)
    right_interp = interp1d(stim_time, right_regressor_raw, kind='nearest', bounds_error=False, fill_value=0)
    
    # Generate regressors at imaging time points
    left_regressor = left_interp(imaging_time)
    right_regressor = right_interp(imaging_time)
    
    return left_regressor, right_regressor, imaging_time

def visualize_regressors(left_regressor, right_regressor, time_points, output_path='motion_regressors.png'):
    """
    Visualize the leftward and rightward motion regressors.
    
    Parameters:
    -----------
    left_regressor : array
        Leftward motion regressor
    right_regressor : array
        Rightward motion regressor
    time_points : array
        Time points for the regressors
    output_path : str
        Path to save the visualization
    """
    import matplotlib.pyplot as plt
    
    plt.figure(figsize=(12, 6))
    
    # Plot left regressor
    plt.subplot(2, 1, 1)
    plt.plot(time_points, left_regressor, 'r-', label='Leftward Motion (Red)')
    plt.xlabel('Time (s)')
    plt.ylabel('Regressor Value')
    plt.title('Leftward Motion Regressor')
    plt.grid(True)
    plt.legend()
    
    # Plot right regressor
    plt.subplot(2, 1, 2)
    plt.plot(time_points, right_regressor, 'k-', label='Rightward Motion (Black)')
    plt.xlabel('Time (s)')
    plt.ylabel('Regressor Value')
    plt.title('Rightward Motion Regressor')
    plt.grid(True)
    plt.legend()
    
    plt.tight_layout()
    plt.savefig(output_path)
    
    # Only show if not running in a script
    if '__file__' not in globals():
        plt.show()
    else:
        plt.close()

def save_regressors(left_regressor, right_regressor, time_points, 
                  output_path='motion_regressors.h5', save_as_npz=False):
    """
    Save the regressors to a file.
    
    Parameters:
    -----------
    left_regressor : array
        Leftward motion regressor
    right_regressor : array
        Rightward motion regressor
    time_points : array
        Time points for the regressors
    output_path : str
        Path to save the regressors
    save_as_npz : bool
        If True, save as .npz file instead of .h5
    """
    if save_as_npz or not (output_path.endswith('.h5') or output_path.endswith('.hdf5')):
        # Save as npz (numpy's compressed format)
        npz_path = output_path if output_path.endswith('.npz') else output_path.replace('.h5', '.npz').replace('.hdf5', '.npz')
        np.savez(npz_path, 
                left_regressor=left_regressor, 
                right_regressor=right_regressor, 
                time_points=time_points)
        print(f"Regressors saved to {npz_path}")
    else:
        # Save as h5 file using flammkuchen
        data_dict = {
            'left_regressor': left_regressor,
            'right_regressor': right_regressor,
            'time_points': time_points,
            'metadata': {
                'imaging_length': len(left_regressor),
                'creation_date': pd.Timestamp.now().isoformat()
            }
        }
        
        fl.save(output_path, data_dict, compression='blosc')
        print(f"Regressors saved to {output_path}")

# Process multiple session folders
def process_all_sessions(base_dir, session_pattern="00*", stimulus_file_pattern="*stimulus_log.hdf5", 
                         imaging_length=1831, imaging_rate=3, dataset_name=None, group_path="/"):
    """
    Process stimulus data from multiple session folders and save regressors.
    
    Parameters:
    -----------
    base_dir : str
        Base directory containing session folders
    session_pattern : str
        Pattern to match session folders (e.g., "00*")
    stimulus_file_pattern : str
        Pattern to match stimulus log files (e.g., "*stimulus_log.hdf5")
    imaging_length : int
        Length of the imaging data (number of frames)
    imaging_rate : float
        Imaging frame rate in Hz
    dataset_name : str or None
        Name of the dataset within the h5 file
    group_path : str
        Path to the group containing the dataset in the h5 file
    """
    # Find all session folders
    session_folders = sorted(glob.glob(os.path.join(base_dir, session_pattern)))
    
    if not session_folders:
        print(f"No session folders found matching pattern '{session_pattern}' in {base_dir}")
        return
    
    print(f"Found {len(session_folders)} session folders")
    
    # Process each session folder
    for session_folder in session_folders:
        session_name = os.path.basename(session_folder)
        print(f"\nProcessing session: {session_name}")
        
        # Find stimulus log file
        stimulus_files = glob.glob(os.path.join(session_folder, stimulus_file_pattern))
        
        if not stimulus_files:
            print(f"  No stimulus log file found in {session_folder}")
            continue
        
        stimulus_file = stimulus_files[0]  # Use the first matching file
        print(f"  Found stimulus log file: {os.path.basename(stimulus_file)}")
        
        try:
            # Build regressors
            left_regressor, right_regressor, time_points = build_motion_regressors(
                stimulus_file,
                imaging_length=imaging_length,
                imaging_rate=imaging_rate,
                dataset_name=dataset_name,
                group_path=group_path
            )
            
            # Create output filenames
            output_h5 = os.path.join(session_folder, "motion_regressors.h5")
            output_npz = os.path.join(session_folder, "motion_regressors.npz")
            output_plot = os.path.join(session_folder, "motion_regressors.png")
            
            # Save regressors
            save_regressors(left_regressor, right_regressor, time_points, output_h5)
            save_regressors(left_regressor, right_regressor, time_points, output_npz, save_as_npz=True)
            
            # Visualize and save plot
            visualize_regressors(left_regressor, right_regressor, time_points, output_path=output_plot)
            
            # Print summary
            print(f"  Created regressors with length: {len(left_regressor)}")
            print(f"  Leftward motion periods: {int(np.sum(left_regressor))} frames")
            print(f"  Rightward motion periods: {int(np.sum(right_regressor))} frames")
            print(f"  Saved regressors to {output_h5} and {output_npz}")
            print(f"  Saved visualization to {output_plot}")
            
        except Exception as e:
            print(f"  Error processing {stimulus_file}: {str(e)}")

In [ ]:
master = Path(r"Z:\Hagar\main\e0020 imaging")

fish_list = list(master.glob("*_v41*"))
fish = fish_list[0]
print(fish)
num_fish = len(fish_list)

In [ ]:
fish_list[6:]

In [ ]:
exp = TwoPExperiment(fish)

In [ ]:
sh = exp.stack_shape
print(sh)

In [ ]:
base_dir = str(fish / 'suite2p')

traces = fl.load(fish / 'suite2p' / '0000' / 'data_from_suite2p_cells.h5')['traces']
len_rec = np.shape(traces)[1]

In [ ]:
for fish in fish_list[6:]:
    
    base_dir = str(fish / 'suite2p')

    traces = fl.load(fish / 'suite2p' / '0000' / 'data_from_suite2p_cells.h5')['traces']
    len_rec = np.shape(traces)[1]

    process_all_sessions(
        base_dir,
        session_pattern="000*",             # Pattern to match session folders (00*)
        stimulus_file_pattern="*stimulus_log.hdf5",  # Pattern to match stimulus log files
        imaging_length=len_rec,                # Length of imaging data
        imaging_rate=3,                     # Imaging at 3 Hz
        dataset_name=None,                  # Name of dataset in h5 file (change as needed)
        group_path="/"                      # Group path in h5 file (change as needed)
    )